In [1]:
import pandas as pd

In [2]:
num_interactions_dict = {
    100_000: '100k',
    500_000: '500k',
    1_000_000: '1M',
    5_000_000: '5M',
    15_000_000: '15M',
    30_000_000: '30M'
}

num_items_dict = {
    1_000: '1k',
    5_000: '5k',
    10_000: '10k',
    50_000: '50k',
    100_000: '100k'
}

In [3]:
algorithms_list = [
    'mab2rec_LinUCB',
    'mab2rec_LinTS',
    'mab2rec_LinGreedy',
    'gobrec_LinUCB_GPU',
    'gobrec_LinTS_GPU',
    'gobrec_LinGreedy_GPU',
    'gobrec_LinUCB_CPU',
    'gobrec_LinTS_CPU',
    'gobrec_LinGreedy_CPU',
    'irec_LinUCB',
    #'irec_LinTS',
    'irec_LinGreedy'
]

In [4]:

from utils.constants import RESULTS_SAVE_PATH, PLOTS_AND_TABLES_SAVE_PATH
import os

def get_experiment_mean_and_std(exp_type: str, algo_name: str, num_interactions_num: int, num_interactions_str: str, num_items_num: int, num_items_str: str):
    df = pd.read_csv(os.path.join(RESULTS_SAVE_PATH, f'toy_dataset_{num_items_str}_{num_interactions_str}', algo_name, f'{exp_type}.csv'))
    mean = df['total'].mean()
    std = df['total'].std()
    return mean, std

def get_experiment_mean_and_std_real_dataset(exp_type: str, algo_name: str, dataset_key: str):
    df = pd.read_csv(os.path.join(RESULTS_SAVE_PATH, dataset_key, algo_name, f'{exp_type}.csv'))
    mean = df['total'].mean()
    std = df['total'].std()
    return mean, std


## 1. Time comparison tables

## 1.1 Time comparison table for toy datasets in LaTeX format

In [ ]:

# ----------------------------------
# CHANGE HERE TO ADD/REMOVE DATASETS
concat_datasets = [
    {
        'num_items': 1_000,
        'num_interactions': 500_000
    },
    {
        'num_items': 10_000,
        'num_interactions': 1_000_000
    },
    {
        'num_items': 100_000,
        'num_interactions': 5_000_000
    }
]
# CHANGE HERE TO ADD/REMOVE EXPERIMENT TYPES
exp_types = [
    'not_incremental',
    'incremental'
]
# ----------------------------------


def generate_time_comparision_table_header():
    return '''
\\begin{table}[htpb]
    \centering
    \\begin{tabular}{l|cccc|cccc|cccc}

    & \multicolumn{4}{c|}{LinGreedy} & \multicolumn{4}{c|}{LinUCB} & \multicolumn{4}{c}{LinTS} \\\\

    \hline
    
    & \makecell{Time\\\\(m)} & \makecell{Opt.\\\\mab2rec} & \makecell{Opt.\\\\iRec} & \makecell{Opt.\\\\CPU} & \makecell{Time\\\\(m)} & \makecell{Opt.\\\\mab2rec} & \makecell{Opt.\\\\iRec} & \makecell{Opt.\\\\CPU} & \makecell{Time\\\\(m)} & \makecell{Opt.\\\\mab2rec} & \makecell{Opt.\\\\iRec} & \makecell{Opt.\\\\CPU} \\\\
    '''

def generate_time_comparision_table_dataset_separation(num_items_num: int, num_items_str: str, num_interactions_num: int, num_interactions_str: str):
    return '\hline\n & \multicolumn{12}{c}{' + f'Toy dataset with {num_items_num:,} ({num_items_str}) items and {num_interactions_num:,} ({num_interactions_str}) interactions' + '} \\\\\n\n\t\hline\n'

def generate_time_comparision_table_footer(label: str, caption_prefix: str = '', caption_posfix: str = ''):
    return f'''
    \end{{tabular}}
    \caption{{{caption_prefix}``Opt. mab2rec'' columns indicates how many times the specified algorithm was faster than the mab2rec implementation. ``Opt. iRec'' columns indicates how many times the specified algorithm was faster than the iRec implementation. ``Opt. CPU'' indicates how many times the specified algorithm was faster than the gobrec CPU implementation.{caption_posfix}}}
    \label{{{label}}}
\end{{table}}
    '''

def generate_time_comparision_table_data(exp_type: str, num_items_num: int, num_items_str: str, num_interactions_num: int, num_interactions_str: str):
    algo_names = [
        'LinGreedy',
        'LinUCB',
        'LinTS'
    ]

    mab2rec_str = '\tmab2rec'
    iRec_str = '\tiRec'
    gobrec_cpu_str = '\t\makecell{gobrec\\\\CPU}'
    gobrec_gpu_str = '\t\makecell{gobrec\\\\GPU}'

    for algo_name in algo_names:
        # mab2rec
        mab2rec_mean, mab2rec_std = get_experiment_mean_and_std(exp_type, f'mab2rec_{algo_name}', num_interactions_num, num_interactions_str, num_items_num, num_items_str)
        mab2rec_mean /= 60  # Convert seconds to minutes
        mab2rec_std /= 60  # Convert seconds to minutes
        mab2rec_str += f' & \makecell{{{mab2rec_mean:.1f}\\\\($\pm$ {mab2rec_std:.2f})}} & -- & -- & --'

        # iRec
        iRec_mean, iRec_std = get_experiment_mean_and_std(exp_type, f'iRec_{algo_name}', num_interactions_num, num_interactions_str, num_items_num, num_items_str)
        iRec_mean /= 60  # Convert seconds to minutes
        iRec_std /= 60  # Convert seconds to minutes
        iRec_str += f' & \makecell{{{iRec_mean:.1f}\\\\($\pm$ {iRec_std:.2f})}} & -- & -- & --'

        # gobrec CPU
        gobrec_cpu_mean, gobrec_cpu_std = get_experiment_mean_and_std(exp_type, f'gobrec_{algo_name}_CPU', num_interactions_num, num_interactions_str, num_items_num, num_items_str)
        gobrec_cpu_mean /= 60  # Convert seconds to minutes
        gobrec_cpu_std /= 60  # Convert seconds to minutes
        gobrec_cpu_str += f' & \makecell{{{gobrec_cpu_mean:.2f}\\\\($\pm$ {gobrec_cpu_std:.2f})}} & {mab2rec_mean / gobrec_cpu_mean:.1f}$\\times$ & {iRec_mean / gobrec_cpu_mean:.1f}$\\times$ & --'

        # gobrec GPU
        gobrec_gpu_mean, gobrec_gpu_std = get_experiment_mean_and_std(exp_type, f'gobrec_{algo_name}_GPU', num_interactions_num, num_interactions_str, num_items_num, num_items_str)
        gobrec_gpu_mean /= 60  # Convert seconds to minutes
        gobrec_gpu_std /= 60  # Convert seconds to minutes
        gobrec_gpu_str += f' & \makecell{{{gobrec_gpu_mean:.2f}\\\\($\pm$ {gobrec_gpu_std:.2f})}} & {mab2rec_mean / gobrec_gpu_mean:.1f}$\\times$ & {iRec_mean / gobrec_gpu_mean:.1f}$\\times$ & {gobrec_cpu_mean / gobrec_gpu_mean:.1f}$\\times$'
    
    return '\n' + mab2rec_str + ' \\\\\n' + iRec_str + ' \\\\\n' + gobrec_cpu_str + ' \\\\\n' + gobrec_gpu_str + ' \\\\\n'

from utils.constants import PLOTS_AND_TABLES_SAVE_PATH
import os

for exp_type in exp_types:
    table_str = ''
    table_str += generate_time_comparision_table_header()

    for dataset in concat_datasets:
        num_items_num = dataset['num_items']
        num_items_str = num_items_dict[num_items_num]
        num_interactions_num = dataset['num_interactions']
        num_interactions_str = num_interactions_dict[num_interactions_num]

        table_str += generate_time_comparision_table_dataset_separation(num_items_num, num_items_str, num_interactions_num, num_interactions_str)
        table_str += generate_time_comparision_table_data(exp_type, num_items_num, num_items_str, num_interactions_num, num_interactions_str)
    
    table_str += generate_time_comparision_table_footer(f'tab:time_comparison_{exp_type}', caption_prefix=f'Execution time comparison for {exp_type.replace("_", " ")} experiments. ')

    os.makedirs(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables'), exist_ok=True)
    with open(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables', f'{exp_type}.tex'), 'w') as f:
        f.write(table_str)


for exp_type in exp_types:
    for num_items_num, num_items_str in num_items_dict.items():
        for num_interactions_num, num_interactions_str in num_interactions_dict.items():
            table_str = ''
            table_str += generate_time_comparision_table_header()
            table_str += '\n\hline\n'
            table_str += generate_time_comparision_table_data(exp_type, num_items_num, num_items_str, num_interactions_num, num_interactions_str)
            table_str += generate_time_comparision_table_footer(f'tab:time_comparison_{exp_type}_{num_items_str}_{num_interactions_str}', caption_prefix=f'Execution time comparison for {exp_type.replace("_", " ")} experiments with {num_items_num:,} ({num_items_str}) items and {num_interactions_num:,} ({num_interactions_str}) interactions. ')

            save_dir = os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables', 'per_dataset')
            os.makedirs(save_dir, exist_ok=True)
            with open(os.path.join(save_dir, f'{exp_type}_{num_items_str}_{num_interactions_str}.tex'), 'w') as f:
                f.write(table_str)

## 1.2 Time comparison table for real datasets in LaTeX format

In [ ]:
# ----------------------------------
# CHANGE HERE TO ADD/REMOVE REAL DATASETS (dataset id, dataset display name)
concat_datasets = [
    ['ml-100k', 'MovieLens 100k'],
    ['ml-1m', 'MovieLens 1M'],
    ['ml-10m', 'MovieLens 10M']
]
# CHANGE HERE TO ADD/REMOVE EXPERIMENT TYPES
exp_types = [
    'not_incremental',
    'incremental'
]
# ----------------------------------


def generate_time_comparision_table_dataset_separation_real_dataset(dataset_name: str):
    return '\hline\n & \multicolumn{12}{c}{' + dataset_name + '} \\\\\n\n\t\hline\n'

def generate_time_comparision_table_data_real_dataset(exp_type: str, dataset_key: str):
    algo_names = [
        'LinGreedy',
        'LinUCB',
        'LinTS'
    ]

    mab2rec_str = '\tmab2rec'
    iRec_str = '\tiRec'
    gobrec_cpu_str = '\t\makecell{gobrec\\\\CPU}'
    gobrec_gpu_str = '\t\makecell{gobrec\\\\GPU}'

    for algo_name in algo_names:
        # mab2rec
        mab2rec_mean, mab2rec_std = get_experiment_mean_and_std_real_dataset(exp_type, f'mab2rec_{algo_name}', dataset_key)
        mab2rec_mean /= 60  # Convert seconds to minutes
        mab2rec_std /= 60  # Convert seconds to minutes
        mab2rec_str += f' & \makecell{{{mab2rec_mean:.1f}\\\\($\pm$ {mab2rec_std:.2f})}} & -- & -- & --'

        # iRec
        iRec_mean, iRec_std = (600, 1) #get_experiment_mean_and_std_real_dataset(exp_type, f'iRec_{algo_name}', dataset_key)
        iRec_mean /= 60  # Convert seconds to minutes
        iRec_std /= 60  # Convert seconds to minutes
        iRec_str += f' & \makecell{{{iRec_mean:.1f}\\\\($\pm$ {iRec_std:.2f})}} & -- & -- & --'

        # gobrec CPU
        gobrec_cpu_mean, gobrec_cpu_std = get_experiment_mean_and_std_real_dataset(exp_type, f'gobrec_{algo_name}_CPU', dataset_key)
        gobrec_cpu_mean /= 60  # Convert seconds to minutes
        gobrec_cpu_std /= 60  # Convert seconds to minutes
        gobrec_cpu_str += f' & \makecell{{{gobrec_cpu_mean:.2f}\\\\($\pm$ {gobrec_cpu_std:.2f})}} & {mab2rec_mean / gobrec_cpu_mean:.1f}$\\times$ & {iRec_mean / gobrec_cpu_mean:.1f}$\\times$ & --'

        # gobrec GPU
        gobrec_gpu_mean, gobrec_gpu_std = get_experiment_mean_and_std_real_dataset(exp_type, f'gobrec_{algo_name}_GPU', dataset_key)
        gobrec_gpu_mean /= 60  # Convert seconds to minutes
        gobrec_gpu_std /= 60  # Convert seconds to minutes
        gobrec_gpu_str += f' & \makecell{{{gobrec_gpu_mean:.2f}\\\\($\pm$ {gobrec_gpu_std:.2f})}} & {mab2rec_mean / gobrec_gpu_mean:.1f}$\\times$ & {iRec_mean / gobrec_gpu_mean:.1f}$\\times$ & {gobrec_cpu_mean / gobrec_gpu_mean:.1f}$\\times$'
    
    return '\n' + mab2rec_str + ' \\\\\n' + iRec_str + ' \\\\\n' + gobrec_cpu_str + ' \\\\\n' + gobrec_gpu_str + ' \\\\\n'

from utils.constants import PLOTS_AND_TABLES_SAVE_PATH
import os

for exp_type in exp_types:
    table_str = ''
    table_str += generate_time_comparision_table_header()

    for dataset_key, dataset_name in concat_datasets:
        num_items_num = dataset['num_items']
        num_items_str = num_items_dict[num_items_num]
        num_interactions_num = dataset['num_interactions']
        num_interactions_str = num_interactions_dict[num_interactions_num]

        table_str += generate_time_comparision_table_dataset_separation_real_dataset(dataset_name)
        table_str += generate_time_comparision_table_data_real_dataset(exp_type, dataset_key)
    
    table_str += generate_time_comparision_table_footer(f'tab:time_comparison_{exp_type}', caption_prefix=f'Execution time comparison for {exp_type.replace("_", " ")} experiments. ')

    os.makedirs(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables'), exist_ok=True)
    with open(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables', f'{exp_type}_real_datasets.tex'), 'w') as f:
        f.write(table_str)


for exp_type in exp_types:
    for dataset_key, dataset_name in concat_datasets:
        table_str = ''
        table_str += generate_time_comparision_table_header()
        table_str += '\n\hline\n'
        table_str += generate_time_comparision_table_data_real_dataset(exp_type, dataset_key)
        table_str += generate_time_comparision_table_footer(f'tab:time_comparison_{exp_type}_{num_items_str}_{num_interactions_str}', caption_prefix=f'Execution time comparison for {exp_type.replace("_", " ")} experiments for dataset {dataset_name}. ')

        save_dir = os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables', 'per_dataset')
        os.makedirs(save_dir, exist_ok=True)
        with open(os.path.join(save_dir, f'{exp_type}_{dataset_key}.tex'), 'w') as f:
            f.write(table_str)

## 1.3 Time comparison table for real datasets in LaTeX format per algorithm

In [ ]:
# ----------------------------------
# CHANGE HERE TO ADD/REMOVE REAL DATASETS (dataset id, dataset display name)
concat_datasets = [
    ['ml-100k', 'MovieLens 100k'],
    ['ml-1m', 'MovieLens 1M'],
    ['ml-10m', 'MovieLens 10M']
]
# CHANGE HERE TO ADD/REMOVE EXPERIMENT TYPES
exp_types = [
    'not_incremental',
    'incremental'
]
# CHANGE HERE TO ADD/REMOVE ALGORITHMS
algo_names = [
    'LinGreedy',
    'LinUCB',
    'LinTS'
]
# ----------------------------------


def generate_time_comparision_table_header_single_algo():
    return '''
\\begin{table}[htpb]
    \centering
    \\begin{tabular}{l|cccc}

    \hline
    
    & \makecell{Time\\\\(m)} & \makecell{Opt.\\\\mab2rec} & \makecell{Opt.\\\\iRec} & \makecell{Opt.\\\\CPU} \\\\
    '''

def generate_time_comparision_table_dataset_separation_real_dataset_single_algo(dataset_name: str):
    return '\hline\n & \multicolumn{4}{c}{' + dataset_name + '} \\\\\n\n\t\hline\n'

def generate_time_comparision_table_data_real_dataset_single_algo(algo_name: str, exp_type: str, dataset_key: str):
    mab2rec_str = '\tmab2rec'
    iRec_str = '\tiRec'
    gobrec_cpu_str = '\t\makecell{gobrec\\\\CPU}'
    gobrec_gpu_str = '\t\makecell{gobrec\\\\GPU}'

    # mab2rec
    mab2rec_mean, mab2rec_std = get_experiment_mean_and_std_real_dataset(exp_type, f'mab2rec_{algo_name}', dataset_key)
    mab2rec_mean /= 60  # Convert seconds to minutes
    mab2rec_std /= 60  # Convert seconds to minutes
    mab2rec_str += f' & \makecell{{{mab2rec_mean:.1f}\\\\($\pm$ {mab2rec_std:.2f})}} & -- & -- & --'

    # iRec
    iRec_mean, iRec_std = (600, 1) #get_experiment_mean_and_std_real_dataset(exp_type, f'iRec_{algo_name}', dataset_key)
    iRec_mean /= 60  # Convert seconds to minutes
    iRec_std /= 60  # Convert seconds to minutes
    iRec_str += f' & \makecell{{{iRec_mean:.1f}\\\\($\pm$ {iRec_std:.2f})}} & -- & -- & --'

    # gobrec CPU
    gobrec_cpu_mean, gobrec_cpu_std = get_experiment_mean_and_std_real_dataset(exp_type, f'gobrec_{algo_name}_CPU', dataset_key)
    gobrec_cpu_mean /= 60  # Convert seconds to minutes
    gobrec_cpu_std /= 60  # Convert seconds to minutes
    gobrec_cpu_str += f' & \makecell{{{gobrec_cpu_mean:.2f}\\\\($\pm$ {gobrec_cpu_std:.2f})}} & {mab2rec_mean / gobrec_cpu_mean:.1f}$\\times$ & {iRec_mean / gobrec_cpu_mean:.1f}$\\times$ & --'

    # gobrec GPU
    gobrec_gpu_mean, gobrec_gpu_std = get_experiment_mean_and_std_real_dataset(exp_type, f'gobrec_{algo_name}_GPU', dataset_key)
    gobrec_gpu_mean /= 60  # Convert seconds to minutes
    gobrec_gpu_std /= 60  # Convert seconds to minutes
    gobrec_gpu_str += f' & \makecell{{{gobrec_gpu_mean:.2f}\\\\($\pm$ {gobrec_gpu_std:.2f})}} & {mab2rec_mean / gobrec_gpu_mean:.1f}$\\times$ & {iRec_mean / gobrec_gpu_mean:.1f}$\\times$ & {gobrec_cpu_mean / gobrec_gpu_mean:.1f}$\\times$'
    
    return '\n' + mab2rec_str + ' \\\\\n' + iRec_str + ' \\\\\n' + gobrec_cpu_str + ' \\\\\n' + gobrec_gpu_str + ' \\\\\n'

from utils.constants import PLOTS_AND_TABLES_SAVE_PATH
import os

for algo_name in algo_names:
    for exp_type in exp_types:
        table_str = ''
        table_str += generate_time_comparision_table_header_single_algo()

        for dataset_key, dataset_name in concat_datasets:
            num_items_num = dataset['num_items']
            num_items_str = num_items_dict[num_items_num]
            num_interactions_num = dataset['num_interactions']
            num_interactions_str = num_interactions_dict[num_interactions_num]

            table_str += generate_time_comparision_table_dataset_separation_real_dataset_single_algo(dataset_name)
            table_str += generate_time_comparision_table_data_real_dataset_single_algo(algo_name, exp_type, dataset_key)
        
        table_str += generate_time_comparision_table_footer(f'tab:time_comparison_{exp_type}', caption_prefix=f'Execution time comparison for {exp_type.replace("_", " ")} {algo_name} experiments. ')

        os.makedirs(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables'), exist_ok=True)
        with open(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables', f'{algo_name}_{exp_type}_real_datasets.tex'), 'w') as f:
            f.write(table_str)


for algo_name in algo_names:
    for exp_type in exp_types:
        for dataset_key, dataset_name in concat_datasets:
            table_str = ''
            table_str += generate_time_comparision_table_header_single_algo()
            table_str += '\n\hline\n'
            table_str += generate_time_comparision_table_data_real_dataset_single_algo(algo_name, exp_type, dataset_key)
            table_str += generate_time_comparision_table_footer(f'tab:time_comparison_{exp_type}_{num_items_str}_{num_interactions_str}', caption_prefix=f'Execution time comparison for {exp_type.replace("_", " ")} {algo_name} experiments for dataset {dataset_name}. ')

            save_dir = os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables', 'per_dataset')
            os.makedirs(save_dir, exist_ok=True)
            with open(os.path.join(save_dir, f'{algo_name}_{exp_type}_{dataset_key}.tex'), 'w') as f:
                f.write(table_str)

## 1.4 Time comparison table for real datasets in MarkDown format per algorithm

In [ ]:

# ----------------------------------
# CHANGE HERE TO ADD/REMOVE REAL DATASETS (dataset id, dataset display name)
concat_datasets = [
    ['ml-100k', 'MovieLens 100k'],
    ['ml-1m', 'MovieLens 1M'],
    ['ml-10m', 'MovieLens 10M']
]
# CHANGE HERE TO ADD/REMOVE EXPERIMENT TYPES
exp_types = [
    'not_incremental',
    'incremental'
]
# CHANGE HERE TO ADD/REMOVE ALGORITHMS
algo_names = [
    'LinGreedy',
    'LinUCB',
    'LinTS'
]
# ----------------------------------


def generate_time_comparision_table_header_single_algo_md():
    return '''
| Algorithm      | Time (m)       | Opt. mab2rec | Opt. iRec | Opt. CPU |
|----------------|----------------|--------------|-----------|----------|
'''

def generate_time_comparision_table_dataset_separation_real_dataset_single_algo_md(dataset_name: str):
    return f'| **{dataset_name}** |                |              |           |          |\n'

def generate_time_comparision_table_data_real_dataset_single_algo_md(algo_name: str, exp_type: str, dataset_key: str):
    mab2rec_str = '| mab2rec        '
    iRec_str = '| iRec           '
    gobrec_cpu_str = '| gobrec CPU     '
    gobrec_gpu_str = '| gobrec GPU     '

    # mab2rec
    mab2rec_mean, mab2rec_std = get_experiment_mean_and_std_real_dataset(exp_type, f'mab2rec_{algo_name}', dataset_key)
    mab2rec_mean /= 60  # Convert seconds to minutes
    mab2rec_std /= 60  # Convert seconds to minutes
    mab2rec_str += f'| {mab2rec_mean:.1f} (± {mab2rec_std:.2f}) |            |         |        |\n'

    # iRec
    iRec_mean, iRec_std = (600, 1) #get_experiment_mean_and_std_real_dataset(exp_type, f'iRec_{algo_name}', dataset_key)
    iRec_mean /= 60  # Convert seconds to minutes
    iRec_std /= 60  # Convert seconds to minutes
    iRec_str += f'| {iRec_mean:.1f} (± {iRec_std:.2f}) |            |         |        |\n'

    # gobrec CPU
    gobrec_cpu_mean, gobrec_cpu_std = get_experiment_mean_and_std_real_dataset(exp_type, f'gobrec_{algo_name}_CPU', dataset_key)
    gobrec_cpu_mean /= 60  # Convert seconds to minutes
    gobrec_cpu_std /= 60  # Convert seconds to minutes
    gobrec_cpu_str += f'| {gobrec_cpu_mean:.1f} (± {gobrec_cpu_std:.2f}) | {mab2rec_mean / gobrec_cpu_mean:.1f} ×    | {iRec_mean / gobrec_cpu_mean:.1f} ×    |        |\n'

    # gobrec GPU
    gobrec_gpu_mean, gobrec_gpu_std = get_experiment_mean_and_std_real_dataset(exp_type, f'gobrec_{algo_name}_GPU', dataset_key)
    gobrec_gpu_mean /= 60  # Convert seconds to minutes
    gobrec_gpu_std /= 60  # Convert seconds to minutes
    gobrec_gpu_str += f'| {gobrec_gpu_mean:.1f} (± {gobrec_gpu_std:.2f}) | {mab2rec_mean / gobrec_gpu_mean:.1f} ×    | {iRec_mean / gobrec_gpu_mean:.1f} ×    | {gobrec_cpu_mean / gobrec_gpu_mean:.1f} ×    |\n'
    
    return mab2rec_str + iRec_str + gobrec_cpu_str + gobrec_gpu_str

from utils.constants import PLOTS_AND_TABLES_SAVE_PATH
import os

for algo_name in algo_names:
    for exp_type in exp_types:
        table_str = ''
        table_str += generate_time_comparision_table_header_single_algo_md()

        for dataset_key, dataset_name in concat_datasets:
            num_items_num = dataset['num_items']
            num_items_str = num_items_dict[num_items_num]
            num_interactions_num = dataset['num_interactions']
            num_interactions_str = num_interactions_dict[num_interactions_num]

            table_str += generate_time_comparision_table_dataset_separation_real_dataset_single_algo_md(dataset_name)
            table_str += generate_time_comparision_table_data_real_dataset_single_algo_md(algo_name, exp_type, dataset_key)

        os.makedirs(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables'), exist_ok=True)
        with open(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables', f'{algo_name}_{exp_type}_real_datasets.md'), 'w') as f:
            f.write(table_str)


for algo_name in algo_names:
    for exp_type in exp_types:
        for dataset_key, dataset_name in concat_datasets:
            table_str = ''
            table_str += generate_time_comparision_table_header_single_algo_md()
            table_str += '\n\hline\n'
            table_str += generate_time_comparision_table_data_real_dataset_single_algo_md(algo_name, exp_type, dataset_key)

            save_dir = os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '1-time_comparision_tables', 'per_dataset')
            os.makedirs(save_dir, exist_ok=True)
            with open(os.path.join(save_dir, f'{algo_name}_{exp_type}_{dataset_key}.md'), 'w') as f:
                f.write(table_str)

## 2. Line plots

In [12]:
import plotly.express as px
import numpy as np
import os

def generate_line_plot(df, x, y, color, log_y, log_x):
    fig = px.line(df, x=x, y=y, color=color, log_y=log_y, log_x=log_x)
    fig.update_layout(
        legend_title_text='',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        margin=dict(b=10, t=0, r=0, l=0)
    )
    #fig.update_xaxes(
    #    tickmode='array',
    #    tickvals=sorted(df['Interactions number'].unique()),
    #    tickangle=45
    #)
    return fig

def generate_line_plot_log2(df, x, y, color, log_y, log_x):

    if log_x:
        df[f'log2_{x}'] = np.log2(df[x])
        x = f'log2_{x}'
    if log_y:
        df[f'log2_{y}'] = np.log2(df[y])
        y = f'log2_{y}'
    
    fig = px.line(df, x=x, y=y, color=color)
    fig.update_layout(
        legend_title_text='',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        margin=dict(b=10, t=0, r=0, l=0)
    )
    return fig

def generate_1_log_line_plot(df, x, y, color):
    df[f'log 1 + {y}'] = np.log1p(df[y])
    fig = px.line(df, x=x, y=f'log 1 + {y}', color=color)
    fig.update_layout(
        legend_title_text='',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        margin=dict(b=10, t=0, r=0, l=0)
    )
    return fig

from plotly.subplots import make_subplots
import plotly.graph_objects as go

def generate_broken_axis_plot(df, x, y, color):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                   row_heights=[0.7, 0.3])
    for color_group in df[color].unique():
        df_group = df[df[color] == color_group]
        # upper range
        fig.add_trace(go.Scatter(x=df_group[x], y=df_group[y], name=color_group), row=1, col=1)
        fig.update_yaxes(range=[50, 10000], row=1, col=1)

        # lower range
        fig.add_trace(go.Scatter(x=df_group[x], y=df_group[y], name=color_group), row=2, col=1)
        fig.update_yaxes(range=[0, 10], row=2, col=1)

    fig.update_layout(
        legend_title_text='',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        margin=dict(b=10, t=0, r=0, l=0)
    )

    return fig

def generate_enhancement_plots(df, x, y, color):

    fig = go.Figure()

    df_gobrecgpu_vs_mab2rec = df[df[color] == 'gobrec GPU'].copy()
    df_gobrecgpu_vs_mab2rec['Enhancement'] = df[df[color] == 'mab2rec'][y].values / df_gobrecgpu_vs_mab2rec[y].values
    fig.add_trace(
        go.Scatter(
            x=df_gobrecgpu_vs_mab2rec[x],
            y=df_gobrecgpu_vs_mab2rec['Enhancement'],
            name='gobrec GPU vs <br>mab2rec',
            showlegend=True
        )
    )

    df_gobreccpu_vs_mab2rec = df[df[color] == 'gobrec CPU'].copy()
    df_gobreccpu_vs_mab2rec['Enhancement'] = df[df[color] == 'mab2rec'][y].values / df_gobreccpu_vs_mab2rec[y].values
    fig.add_trace(
        go.Scatter(
            x=df_gobreccpu_vs_mab2rec[x],
            y=df_gobreccpu_vs_mab2rec['Enhancement'],
            name='gobrec CPU vs <br>mab2rec',
            showlegend=True
        )
    )

    df_gobrecgpu_vs_irec = df[df[color] == 'gobrec GPU'].copy()
    df_gobrecgpu_vs_irec['Enhancement'] = df[df[color] == 'irec'][y].values / df_gobrecgpu_vs_irec[y].values
    fig.add_trace(
        go.Scatter(
            x=df_gobrecgpu_vs_irec[x],
            y=df_gobrecgpu_vs_irec['Enhancement'],
            name='gobrec GPU vs <br>iRec',
            showlegend=True
        )
    )

    df_gobreccpu_vs_irec = df[df[color] == 'gobrec CPU'].copy()
    df_gobreccpu_vs_irec['Enhancement'] = df[df[color] == 'irec'][y].values / df_gobreccpu_vs_irec[y].values
    fig.add_trace(
        go.Scatter(
            x=df_gobreccpu_vs_irec[x],
            y=df_gobreccpu_vs_irec['Enhancement'],
            name='gobrec CPU vs <br>iRec',
            showlegend=True
        )
    )

    fig.update_layout(
        legend_title_text='',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        margin=dict(b=10, t=0, r=0, l=0)
    )

    return fig

enhances_text_pos_map = {
    'interactions': {
        'LinGreedy': {
            'gobrec GPU': ["top center", "bottom center"],
            'gobrec CPU': ["top center", "top center"],
            'irec': ["top center", "top center"],
            'mab2rec': ["bottom center", "bottom center"]
        },
        'LinUCB': {
            'gobrec GPU': ["top center", "top center"],
            'gobrec CPU': ["top center", "top center"],
            'irec': ["top center", "top center"],
            'mab2rec': ["bottom center", "bottom center"]
        }
    },
    'items': {
        'LinGreedy': {
            'gobrec GPU': ["top center", "bottom center"],
            'gobrec CPU': ["top center", "top center"],
            'irec': ["bottom center", "bottom center"],
            'mab2rec': ["top center", "top center"]
        },
        'LinUCB': {
            'gobrec GPU': ["top center", "top center"],
            'gobrec CPU': ["top center", "top center"],
            'irec': ["top center", "top center"],
            'mab2rec': ["bottom center", "bottom center"]
        }
    }
}

def generate_line_plot_with_enhances(df, x, y, color, algo_name, x_type):
    colors = {
        "mab2rec": "#9ec2d7",     # light blue
        "gobrec GPU": "#ffbeb0",  # soft red
        "gobrec CPU": "#bdd0bd",  # light green
        "irec": "#bcbddc"         # light purple
    }
    
    text_colors = {
        "mab2rec": "#0534aa",     # light blue
        "gobrec GPU": "#e50202",  # soft red
        "gobrec CPU": "#277C08",  # light green
        "irec": "#9b1b9f"         # light purple
    }

    fig = go.Figure()
    
    df['enhance'] = (
        df[y] /
        df.groupby(x)
        .apply(lambda g: g.loc[g[color] == "gobrec GPU", y].iloc[0])
        .reindex(df[x]).values
    )
    
    df_group = df[df[color] == 'gobrec GPU']
    
    fig.add_vline(x=df_group[x].iloc[1], line=dict(color="#dad1af", dash='dash'), layer='below')
    fig.add_vline(x=df_group[x].iloc[3], line=dict(color='#dad1af', dash='dash'), layer='below')
    
    for c_group in df[color].unique():
        df_group = df[df[color] == c_group]

        fig.add_trace(
            go.Scatter(
                x=df_group[x],
                y=df_group[y],
                name=c_group,
                showlegend=True,
                mode='lines',
                line=dict(color=colors.get(c_group, 'black'))
            )
        )

    for c_group in [
        'mab2rec', 'irec', 'gobrec CPU', 'gobrec GPU'
    ]:
        df_group = df[df[color] == c_group]

        fig.add_trace(
            go.Scatter(
                x=[df_group[x].iloc[1]],
                y=[df_group[y].iloc[1]],
                mode="markers+text",
                text=[f"{round(df_group[y].iloc[1])}s <br> ({round(df_group['enhance'].iloc[1], 1)}x)"],
                textposition=enhances_text_pos_map[x_type][algo_name][c_group][0],
                showlegend=False,
                textfont=dict(color=text_colors.get(c_group, 'black')),
                line=dict(color=colors.get(c_group, 'black'))
            )
        )

        fig.add_trace(
            go.Scatter(
                x=[df_group[x].iloc[3]],
                y=[df_group[y].iloc[3]],
                mode="markers+text",
                text=[f"{round(df_group[y].iloc[3])}s <br> ({round(df_group['enhance'].iloc[3], 1)}x)"],
                textposition=enhances_text_pos_map[x_type][algo_name][c_group][1],
                showlegend=False,
                textfont=dict(color=text_colors.get(c_group, 'black')),
                line=dict(color=colors.get(c_group, 'black'))
            )
        )
    
    max_y = df[y].max()
    min_y = df[y].min()
    fig.update_yaxes(
        range=[
            np.log10(min_y) * 0.1,
            np.log10(max_y) * 1.2
        ]
    )
    fig.update_xaxes(
        range=[
            df_group[x].min() * 0.9,
            df_group[x].max() * 1.1
        ]
    )
    fig.update_layout(
        legend_title_text='',
        yaxis_type="log",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.2,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        margin=dict(b=10, t=0, r=0, l=0)
    )

    return fig

def generate_line_plot_with_enhances_latex(df, x, y, color, algo_name, x_type):
    colors = {
        "mab2rec": "#9ec2d7",     # light blue
        "gobrec GPU": "#ffbeb0",  # soft red
        "gobrec CPU": "#bdd0bd",  # light green
        "irec": "#bcbddc"         # light purple
    }
    
    text_colors = {
        "mab2rec": "#0534aa",     # light blue
        "gobrec GPU": "#e50202",  # soft red
        "gobrec CPU": "#277C08",  # light green
        "irec": "#9b1b9f"         # light purple
    }

    fig = go.Figure()
    
    df['enhance'] = (
        df[y] /
        df.groupby(x)
        .apply(lambda g: g.loc[g[color] == "gobrec GPU", y].iloc[0])
        .reindex(df[x]).values
    )
    
    df_group = df[df[color] == 'gobrec GPU']
    
    fig.add_vline(x=df_group[x].iloc[1], line=dict(color="#dad1af", dash='dash'), layer='below')
    fig.add_vline(x=df_group[x].iloc[3], line=dict(color='#dad1af', dash='dash'), layer='below')
    
    for c_group in df[color].unique():
        df_group = df[df[color] == c_group]

        fig.add_trace(
            go.Scatter(
                x=df_group[x],
                y=df_group[y],
                name=c_group,
                showlegend=True,
                mode='lines',
                line=dict(color=colors.get(c_group, 'black'))
            )
        )

    for c_group in [
        'mab2rec', 'irec', 'gobrec CPU', 'gobrec GPU'
    ]:
        df_group = df[df[color] == c_group]

        fig.add_trace(
            go.Scatter(
                x=[df_group[x].iloc[1]],
                y=[df_group[y].iloc[1]],
                mode="markers+text",
                text=[f"{round(df_group[y].iloc[1])}s \\\\<br> ({round(df_group['enhance'].iloc[1], 1)}x)"],
                textposition=enhances_text_pos_map[x_type][algo_name][c_group][0],
                showlegend=False,
                textfont=dict(color=text_colors.get(c_group, 'black')),
                line=dict(color=colors.get(c_group, 'black'))
            )
        )

        fig.add_trace(
            go.Scatter(
                x=[df_group[x].iloc[3]],
                y=[df_group[y].iloc[3]],
                mode="markers+text",
                text=[f"{round(df_group[y].iloc[3])}s \\\\<br> ({round(df_group['enhance'].iloc[3], 1)}x)"],
                textposition=enhances_text_pos_map[x_type][algo_name][c_group][1],
                showlegend=False,
                textfont=dict(color=text_colors.get(c_group, 'black')),
                line=dict(color=colors.get(c_group, 'black'))
            )
        )
    
    max_y = df[y].max()
    min_y = df[y].min()
    fig.update_yaxes(
        range=[
            np.log10(min_y) * 0.1,
            np.log10(max_y) * 1.2
        ]
    )
    fig.update_xaxes(
        range=[
            df_group[x].min() * 0.9,
            df_group[x].max() * 1.1
        ]
    )
    fig.update_layout(
        legend_title_text='',
        yaxis_type="log",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.2,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        margin=dict(b=10, t=0, r=0, l=0)
    )

    return fig

from scipy.optimize import nnls


preds_text_pos_map = {
    'interactions': {
        'LinGreedy': {
            'gobrec GPU': ["bottom center", "bottom center"],
            'gobrec CPU': ["top center", "top center"],
            'irec': ["top center", "top center"],
            'mab2rec': ["bottom center", "bottom center"]
        },
        'LinUCB': {
            'gobrec GPU': ["top center", "top center"],
            'gobrec CPU': ["top center", "top center"],
            'irec': ["top center", "top center"],
            'mab2rec': ["bottom center", "bottom center"]
        }
    },
    'items': {
        'LinGreedy': {
            'gobrec GPU': ["bottom center", "bottom center"],
            'gobrec CPU': ["top center", "top center"],
            'irec': ["bottom center", "top center"],
            'mab2rec': ["top center", "top center"]
        },
        'LinUCB': {
            'gobrec GPU': ["top center", "top center"],
            'gobrec CPU': ["top center", "top center"],
            'irec': ["top center", "top center"],
            'mab2rec': ["bottom center", "bottom center"]
        }
    }
}

def generate_predictions_line_plot(df, x, y, color, algo_name, x_type):
    

    def solve(a, b, y_target):
        if a == 0:
            return y_target / b

        discriminant = b**2 + 4*a*y_target
        
        sqrt_disc = np.sqrt(discriminant)
        
        x1 = (-b + sqrt_disc) / (2*a)
        x2 = (-b - sqrt_disc) / (2*a)

        return max(x1, x2)
    
    colors = {
        "mab2rec": "#9ec2d7",     # light blue
        "gobrec GPU": "#ffbeb0",  # soft red
        "gobrec CPU": "#bdd0bd",  # light green
        "irec": "#bcbddc"         # light purple
    }
    
    text_colors = {
        "mab2rec": "#0534aa",     # light blue
        "gobrec GPU": "#e50202",  # soft red
        "gobrec CPU": "#277C08",  # light green
        "irec": "#9b1b9f"         # light purple
    }

    df_group = df[df[color] == 'gobrec GPU']

    w, _ = nnls(
        np.hstack(
            (
                (df_group[x].values ** 2).reshape(-1, 1),
                df_group[x].values.reshape(-1, 1)
            )
        ),
        df_group[y].values
    )

    a, b = w

    one_minute_pred = solve(a, b, 60)
    one_hour_pred = solve(a, b, 60*60)

    items_between = np.arange(one_minute_pred, one_hour_pred, (one_hour_pred - one_minute_pred) / 10)
    items_end = [one_hour_pred, one_hour_pred * 1.03, one_hour_pred * 1.06, one_hour_pred * 1.09, one_hour_pred * 1.12, one_hour_pred * 1.15, one_hour_pred * 1.18]
    new_x = np.concatenate(([one_minute_pred], items_between, [one_hour_pred], items_end))
    
    fig = go.Figure()
    
    # Add vlines to indicate 1 minute and 1 hour
    fig.add_vline(x=one_minute_pred, line=dict(color="#dad1af", dash='dash'), layer='below')
    fig.add_vline(x=one_hour_pred, line=dict(color='#dad1af', dash='dash'), layer='below')
    
    min_y = 10000000
    max_y = 0
    
    for c_group in df[color].unique():
        df_group = df[df[color] == c_group]

        w, _ = nnls(
            np.hstack(
                (
                    (df_group[x].values ** 2).reshape(-1, 1),
                    df_group[x].values.reshape(-1, 1)
                )
            ),
            df_group[y].values
        )
        a, b = w
        
        predictions = a * (new_x ** 2) + b * new_x
        fig.add_trace(
            go.Scatter(
                x=new_x,
                y=predictions,
                name=c_group,
                showlegend=True,
                mode='lines',
                line=dict(color=colors.get(c_group, 'black'))
            )
        )
        
        min_y = min(min_y, predictions.min())
        max_y = max(max_y, predictions.max())

    for c_group in df[color].unique():
        df_group = df[df[color] == c_group]

        w, _ = nnls(
            np.hstack(
                (
                    (df_group[x].values ** 2).reshape(-1, 1),
                    df_group[x].values.reshape(-1, 1)
                )
            ),
            df_group[y].values
        )
        a, b = w
        y_minute = a * (one_minute_pred ** 2) + b * one_minute_pred
        y_hour = a * (one_hour_pred ** 2) + b * one_hour_pred

        fig.add_trace(
            go.Scatter(
                x=[one_minute_pred],
                y=[y_minute],
                mode="markers+text",
                text=[f"{round(y_minute / 60)}m"],
                textposition=preds_text_pos_map[x_type][algo_name][c_group][0],
                showlegend=False,
                textfont=dict(color=text_colors.get(c_group, 'black')),
                line=dict(color=colors.get(c_group, 'black'))
            )
        )
        
        fig.add_trace(
            go.Scatter(
                x=[one_hour_pred],
                y=[y_hour],
                mode="markers+text",
                text=[f"{round(y_hour / 3600)}h"],
                textposition=preds_text_pos_map[x_type][algo_name][c_group][1],
                showlegend=False,
                textfont=dict(color=text_colors.get(c_group, 'black')),
                line=dict(color=colors.get(c_group, 'black'))
            )
        )
    
    fig.update_layout(
        legend_title_text='',
        yaxis_type="log",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.2,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        margin=dict(b=10, t=0, r=0, l=0)
    )
    
    fig.update_yaxes(
        range=[
            np.log10(min_y) * 0.7,
            np.log10(max_y) * 1.1
        ]
    )
    
    return fig


def generate_latex_figure_line_plot_svg_interactions(exp_type: str, num_items_num: int, num_items_str: str, plot_option_name: str):
    return f'''
\\begin{{figure}}[htbp]
    \centering
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includesvg[width=\\textwidth]{{plots_and_tables/2_1-interactions_line_plots/{exp_type}/LinGreedy/LinGreedy_{num_items_str}_{plot_option_name}}}
        \caption{{LinGreedy}}
    \end{{subfigure}}
    \hfill
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includesvg[width=\\textwidth]{{plots_and_tables/2_1-interactions_line_plots/{exp_type}/LinUCB/LinUCB_{num_items_str}_{plot_option_name}}}
        \caption{{LinUCB}}
    \end{{subfigure}}
    \hfill
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includesvg[width=\\textwidth]{{plots_and_tables/2_1-interactions_line_plots/{exp_type}/LinTS/LinTS_{num_items_str}_{plot_option_name}}}
        \caption{{LinTS}}
    \end{{subfigure}}
    \caption{{Execution time comparison on toy datasets {exp_type.replace('_', ' ')} experiments with {num_items_num:,} ({num_items_str}) items.}}
\end{{figure}}
    '''

def generate_latex_figure_line_plot_jpg_interactions(exp_type: str, num_items_num: int, num_items_str: str, plot_option_name: str):
    return f'''
\\begin{{figure}}[htbp]
    \centering
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includegraphics[width=\\textwidth]{{plots_and_tables/2_1-interactions_line_plots/{exp_type}/LinGreedy/LinGreedy_{num_items_str}_{plot_option_name}}}
        \caption{{LinGreedy}}
    \end{{subfigure}}
    \hfill
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includegraphics[width=\\textwidth]{{plots_and_tables/2_1-interactions_line_plots/{exp_type}/LinUCB/LinUCB_{num_items_str}_{plot_option_name}}}
        \caption{{LinUCB}}
    \end{{subfigure}}
    \hfill
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includegraphics[width=\\textwidth]{{plots_and_tables/2_1-interactions_line_plots/{exp_type}/LinTS/LinTS_{num_items_str}_{plot_option_name}}}
        \caption{{LinTS}}
    \end{{subfigure}}
    \caption{{Execution time comparison on toy datasets {exp_type.replace('_', ' ')} experiments with {num_items_num:,} ({num_items_str}) items.}}
\end{{figure}}
    '''


def generate_latex_figure_line_plot_svg_items(exp_type: str, num_interactions_num: int, num_interactions_str: str, plot_option_name: str):
    return f'''
\\begin{{figure}}[htbp]
    \centering
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includesvg[width=\\textwidth]{{plots_and_tables/2_2-items_line_plots/{exp_type}/LinGreedy/LinGreedy_{num_interactions_str}_{plot_option_name}}}
        \caption{{LinGreedy}}
    \end{{subfigure}}
    \hfill
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includesvg[width=\\textwidth]{{plots_and_tables/2_2-items_line_plots/{exp_type}/LinUCB/LinUCB_{num_interactions_str}_{plot_option_name}}}
        \caption{{LinUCB}}
    \end{{subfigure}}
    \hfill
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includesvg[width=\\textwidth]{{plots_and_tables/2_2-items_line_plots/{exp_type}/LinTS/LinTS_{num_interactions_str}_{plot_option_name}}}
        \caption{{LinTS}}
    \end{{subfigure}}
    \caption{{Execution time comparison on toy datasets {exp_type.replace('_', ' ')} experiments with {num_interactions_num:,} ({num_interactions_str}) interactions.}}
\end{{figure}}
    '''

def generate_latex_figure_line_plot_jpg_items(exp_type: str, num_interactions_num: int, num_interactions_str: str, plot_option_name: str):
    return f'''
\\begin{{figure}}[htbp]
    \centering
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includegraphics[width=\\textwidth]{{plots_and_tables/2_2-items_line_plots/{exp_type}/LinGreedy/LinGreedy_{num_interactions_str}_{plot_option_name}}}
        \caption{{LinGreedy}}
    \end{{subfigure}}
    \hfill
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includegraphics[width=\\textwidth]{{plots_and_tables/2_2-items_line_plots/{exp_type}/LinUCB/LinUCB_{num_interactions_str}_{plot_option_name}}}
        \caption{{LinUCB}}
    \end{{subfigure}}
    \hfill
    \\begin{{subfigure}}[b]{{0.33\\textwidth}}
        \includegraphics[width=\\textwidth]{{plots_and_tables/2_2-items_line_plots/{exp_type}/LinTS/LinTS_{num_interactions_str}_{plot_option_name}}}
        \caption{{LinTS}}
    \end{{subfigure}}
    \caption{{Execution time comparison on toy datasets {exp_type.replace('_', ' ')} experiments with {num_interactions_num:,} ({num_interactions_str}) interactions.}}
\end{{figure}}
    '''



In [13]:
# ----------------------------------
# CHANGE HERE TO ADD/REMOVE ALGORITHMS
algo_names = [
    'LinGreedy',
    'LinUCB',
    #'LinTS'
]
# CHANGE HERE TO ADD/REMOVE PLOTS OPTIONS
plot_options = {
    'ori': lambda df, x, y, color, algo_name: generate_line_plot(df, x, y, color, log_y=False, log_x=False),

    'log10y': lambda df, x, y, color, algo_name: generate_line_plot(df, x, y, color, log_y=True, log_x=False),
    #'log10x': lambda df, x, y, color, algo_name: generate_line_plot(df, x, y, color, log_y=False, log_x=True),
    #'log10x_log10y': lambda df, x, y, color, algo_name: generate_line_plot(df, x, y, color, log_y=True, log_x=True),

    'log2y': lambda df, x, y, color, algo_name: generate_line_plot_log2(df, x, y, color, log_y=True, log_x=False),
    #'log2x': lambda df, x, y, color, algo_name: generate_line_plot_log2(df, x, y, color, log_y=False, log_x=True),
    #'log2x_log2y': lambda df, x, y, color, algo_name: generate_line_plot_log2(df, x, y, color, log_y=True, log_x=True),

    'log1py': lambda df, x, y, color, algo_name: generate_1_log_line_plot(df, x, y, color),

    'broken_axis': lambda df, x, y, color, algo_name: generate_broken_axis_plot(df, x, y, color),

    'enhance': lambda df, x, y, color, algo_name: generate_enhancement_plots(df, x, y, color),
    
    'pred': lambda df, x, y, color, algo_name: generate_predictions_line_plot(df, x, y, color, algo_name, 'interactions'),
    
    'enhance_and_time': lambda df, x, y, color, algo_name: generate_line_plot_with_enhances(df, x, y, color, algo_name, 'interactions'),
    
    'enhance_and_time_latex': lambda df, x, y, color, algo_name: generate_line_plot_with_enhances_latex(df, x, y, color, algo_name, 'interactions')
}
# CHANGE HERE TO ADD/REMOVE EXPERIMENT TYPES
exp_types = [
    # 'not_incremental',
    'incremental'
]
# CHANGE HERE TO ADD/REMOVE NUMBER OF ITEMS
num_items = [
    1_000,
    #10_000,
    #100_000
]
# CHANGE HERE TO ADD/REMOVE NUMBER OF INTERACTIONS
num_interactions = [
    100_000,
    500_000,
    1_000_000,
    5_000_000,
    #15_000_000,
    #30_000_000
]
# ----------------------------------

import plotly.express as px
import os

for algo_name in algo_names:
    for plot_option_name, plot_option_func in plot_options.items():
        for exp_type in exp_types:
            for num_items_num in num_items:
                num_items_str = num_items_dict[num_items_num]
                data = []
                for num_interactions_num in num_interactions:
                    num_interactions_str = num_interactions_dict[num_interactions_num]
                    for full_algo_name in algorithms_list:
                        if algo_name in full_algo_name:
                            mean, std = get_experiment_mean_and_std(exp_type, full_algo_name, num_interactions_num, num_interactions_str, num_items_num, num_items_str)
                            data.append({
                                'Algorithm': full_algo_name.replace(f'_{algo_name}', '').replace('_', ' '),
                                'Interactions number': num_interactions_num,
                                'Mean execution time (s)': mean,
                                'Standard deviation (s)': std
                            })
                    
                df = pd.DataFrame(data)
                save_path = os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '2_1-interactions_line_plots', exp_type, algo_name)
                fig = plot_option_func(df, x='Interactions number', y='Mean execution time (s)', color='Algorithm', algo_name=algo_name)
                save_path = os.path.join(save_path, f'{algo_name}_{num_items_str}_{plot_option_name}')
                os.makedirs(os.path.dirname(save_path), exist_ok=True)
                if not 'latex' in plot_option_name:
                    fig.write_image(f'{save_path}.jpg', width=380, height=380)
                fig.write_image(f'{save_path}.svg', width=300, height=300)

for exp_type in exp_types:
    for num_items_num, num_items_str in num_items_dict.items():
        for plot_option_name in plot_options.keys():
            if num_items_num in num_items:
                with open(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '2_1-interactions_line_plots', exp_type, f'{num_items_str}_{plot_option_name}_jpg.tex'), 'w') as f:
                    f.write(generate_latex_figure_line_plot_jpg_interactions(exp_type, num_items_num, num_items_str, plot_option_name))

                with open(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '2_1-interactions_line_plots', exp_type, f'{num_items_str}_{plot_option_name}_svg.tex'), 'w') as f:
                    f.write(generate_latex_figure_line_plot_svg_interactions(exp_type, num_items_num, num_items_str, plot_option_name))
                    

In [14]:
# ----------------------------------
# CHANGE HERE TO ADD/REMOVE ALGORITHMS
algo_names = [
    'LinGreedy',
    'LinUCB',
    #'LinTS'
]
# CHANGE HERE TO ADD/REMOVE PLOTS OPTIONS
plot_options = {
    'ori': lambda df, x, y, color, algo_name: generate_line_plot(df, x, y, color, log_y=False, log_x=False),

    'log10y': lambda df, x, y, color, algo_name: generate_line_plot(df, x, y, color, log_y=True, log_x=False),
    #'log10x': lambda df, x, y, color, algo_name: generate_line_plot(df, x, y, color, log_y=False, log_x=True),
    #'log10x_log10y': lambda df, x, y, color, algo_name: generate_line_plot(df, x, y, color, log_y=True, log_x=True),

    'log2y': lambda df, x, y, color, algo_name: generate_line_plot_log2(df, x, y, color, log_y=True, log_x=False),
    #'log2x': lambda df, x, y, color, algo_name: generate_line_plot_log2(df, x, y, color, log_y=False, log_x=True),
    #'log2x_log2y': lambda df, x, y, color, algo_name: generate_line_plot_log2(df, x, y, color, log_y=True, log_x=True),

    'log1py': lambda df, x, y, color, algo_name: generate_1_log_line_plot(df, x, y, color),

    'broken_axis': lambda df, x, y, color, algo_name: generate_broken_axis_plot(df, x, y, color),

    'enhance': lambda df, x, y, color, algo_name: generate_enhancement_plots(df, x, y, color),
    
    'pred': lambda df, x, y, color, algo_name: generate_predictions_line_plot(df, x, y, color, algo_name, 'items'),
    
    'enhance_and_time': lambda df, x, y, color, algo_name: generate_line_plot_with_enhances(df, x, y, color, algo_name, 'items'),
    
    'enhance_and_time_latex': lambda df, x, y, color, algo_name: generate_line_plot_with_enhances_latex(df, x, y, color, algo_name, 'items')
}

# CHANGE HERE TO ADD/REMOVE EXPERIMENT TYPES
exp_types = [
    # 'not_incremental',
    'incremental'
]
# CHANGE HERE TO ADD/REMOVE NUMBER OF ITEMS
num_items = [
    1_000,
    5_000,
    10_000,
    50_000,
    #100_000
]
# CHANGE HERE TO ADD/REMOVE NUMBER OF INTERACTIONS
num_interactions = [
    #100_000,
    500_000,
    #1_000_000,
    #5_000_000,
    #15_000_000,
    #30_000_000
]
# ----------------------------------

import plotly.express as px
import os

for algo_name in algo_names:
    for plot_option_name, plot_option_func in plot_options.items():
        for exp_type in exp_types:
            for num_interactions_num in num_interactions:
                num_interactions_str = num_interactions_dict[num_interactions_num]
                data = []
                for num_items_num in num_items:
                    num_items_str = num_items_dict[num_items_num]
                    for full_algo_name in algorithms_list:
                        if algo_name in full_algo_name:
                            mean, std = get_experiment_mean_and_std(exp_type, full_algo_name, num_interactions_num, num_interactions_str, num_items_num, num_items_str)
                            data.append({
                                'Algorithm': full_algo_name.replace(f'_{algo_name}', '').replace('_', ' '),
                                'Items number': num_items_num,
                                'Mean execution time (s)': mean,
                                'Standard deviation (s)': std
                            })
                    
                df = pd.DataFrame(data)
                save_path = os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '2_2-items_line_plots', exp_type, algo_name)
                fig = plot_option_func(df, x='Items number', y='Mean execution time (s)', color='Algorithm', algo_name=algo_name)
                save_path = os.path.join(save_path, f'{algo_name}_{num_interactions_str}_{plot_option_name}')
                os.makedirs(os.path.dirname(save_path), exist_ok=True)
                if not 'latex' in plot_option_name:
                    fig.write_image(f'{save_path}.jpg', width=380, height=380)
                fig.write_image(f'{save_path}.svg', width=300, height=300)

for exp_type in exp_types:
    for num_interactions_num, num_interactions_str in num_interactions_dict.items():
        for plot_option_name in plot_options.keys():
            if num_interactions_num in num_interactions:
                with open(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '2_2-items_line_plots', exp_type, f'{num_interactions_str}_{plot_option_name}_jpg.tex'), 'w') as f:
                    f.write(generate_latex_figure_line_plot_jpg_items(exp_type, num_interactions_num, num_interactions_str, plot_option_name))

                with open(os.path.join(PLOTS_AND_TABLES_SAVE_PATH, '2_2-items_line_plots', exp_type, f'{num_interactions_str}_{plot_option_name}_svg.tex'), 'w') as f:
                    f.write(generate_latex_figure_line_plot_svg_items(exp_type, num_interactions_num, num_interactions_str, plot_option_name))

In [125]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

def generate_broken_axis_plot(df, x, y, color):
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.7, 0.3]
    )

    color_map = {
        group: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
        for i, group in enumerate(df[color].unique())
    }

    for color_group in df[color].unique():
        df_group = df[df[color] == color_group]
        c = color_map[color_group]

        fig.add_trace(
            go.Scatter(
                x=df_group[x],
                y=df_group[y],
                name=color_group,
                legendgroup=color_group,
                line=dict(color=c),
                showlegend=True
            ),
            row=1, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=df_group[x],
                y=df_group[y],
                name=color_group,
                legendgroup=color_group,
                line=dict(color=c),
                showlegend=False
            ),
            row=2, col=1
        )

    fig.update_yaxes(range=[1_000, 30_000], row=1, col=1)
    fig.update_yaxes(range=[0, 30], row=2, col=1)

    fig.update_layout(
        legend_title_text='',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        margin=dict(b=10, t=0, r=0, l=0)
    )

    return fig

generate_broken_axis_plot(df, x='Items number', y='Mean execution time (s)', color='Algorithm')

In [ ]:
import plotly.graph_objects as go

def generate_enhancement_plots(df, x, y, color):

    fig = go.Figure()

    df_gobrecgpu_vs_mab2rec = df[df[color] == 'gobrec GPU'].copy()
    df_gobrecgpu_vs_mab2rec['Enhancement'] = df[df[color] == 'mab2rec'][y].values / df_gobrecgpu_vs_mab2rec[y].values
    fig.add_trace(
        go.Scatter(
            x=df_gobrecgpu_vs_mab2rec[x],
            y=df_gobrecgpu_vs_mab2rec['Enhancement'],
            name='gobrec GPU vs <br>mab2rec',
            showlegend=True
        )
    )

    df_gobreccpu_vs_mab2rec = df[df[color] == 'gobrec CPU'].copy()
    df_gobreccpu_vs_mab2rec['Enhancement'] = df[df[color] == 'mab2rec'][y].values / df_gobreccpu_vs_mab2rec[y].values
    fig.add_trace(
        go.Scatter(
            x=df_gobreccpu_vs_mab2rec[x],
            y=df_gobreccpu_vs_mab2rec['Enhancement'],
            name='gobrec CPU vs <br>mab2rec',
            showlegend=True
        )
    )

    df_gobrecgpu_vs_irec = df[df[color] == 'gobrec GPU'].copy()
    df_gobrecgpu_vs_irec['Enhancement'] = df[df[color] == 'irec'][y].values / df_gobrecgpu_vs_irec[y].values
    fig.add_trace(
        go.Scatter(
            x=df_gobrecgpu_vs_irec[x],
            y=df_gobrecgpu_vs_irec['Enhancement'],
            name='gobrec GPU vs <br>iRec',
            showlegend=True
        )
    )

    df_gobreccpu_vs_irec = df[df[color] == 'gobrec CPU'].copy()
    df_gobreccpu_vs_irec['Enhancement'] = df[df[color] == 'irec'][y].values / df_gobreccpu_vs_irec[y].values
    fig.add_trace(
        go.Scatter(
            x=df_gobreccpu_vs_irec[x],
            y=df_gobreccpu_vs_irec['Enhancement'],
            name='gobrec CPU vs <br>iRec',
            showlegend=True
        )
    )

    fig.update_layout(
        legend_title_text='',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        margin=dict(b=10, t=0, r=0, l=0)
    )

    return fig

generate_enhancement_plots(df, x='Items number', y='Mean execution time (s)', color='Algorithm')

In [ ]:
from gobrec.mabs.lin_mabs import Lin

lin = Lin(l2_lambda=0)

lin.fit(
    contexts=np.array([
        [1, 1],
        [4, 2],
        [9, 3],
        [16, 4],
        [25, 5]
    ]),
    decisions=np.array([1,1,1,1,1]),
    rewards=np.array([2,6,12,20,30])
)

print(lin.predict(np.array([[36, 6]])))
print(lin.beta.numpy())

In [ ]:
from scipy.optimize import nnls

nnls(
    np.array([
        [1, 1],
        [4, 2],
        [9, 3],
        [16, 4],
        [25, 5]
    ]),
    np.array([2,6,12,20,30])
)

In [9]:
from gobrec.mabs.lin_mabs import Lin

def generate_predictions_line_plot(df, x, y, color):
    
    fig = go.Figure()
    
    for c_group in df[color].unique():
        df_group = df[df[color] == c_group]
        
        lin = Lin(l2_lambda=0)

        lin.fit(
            contexts=np.hstack(
                (
                    (df_group[x].values ** 2).reshape(-1, 1),
                    df_group[x].values.reshape(-1, 1)
                )
            ),
            decisions=np.ones(len(df_group)),
            rewards=df_group[y].values
        )

        predictions = lin.predict(
            np.hstack(
                (
                    (df_group[x].values ** 2).reshape(-1, 1),
                    df_group[x].values.reshape(-1, 1)
                )
            )
        ).numpy().flatten()

        fig.add_trace(
            go.Scatter(
                x=df_group[x],
                y=df_group[y],
                name=c_group,
                showlegend=True
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df_group[x],
                y=predictions,
                name=f'{c_group} predictions',
                showlegend=True
            )
        )


    fig.update_layout(
        legend_title_text='',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        margin=dict(b=10, t=0, r=0, l=0)
    )
    return fig

generate_predictions_line_plot(df, x='Items number', y='Mean execution time (s)', color='Algorithm')

ModuleNotFoundError: No module named 'gobrec'

In [ ]:
from scipy.optimize import nnls

def generate_predictions_line_plot(df, x, y, color):

    def solve(a, b, y_target):
        if a == 0:
            return y_target / b

        discriminant = b**2 + 4*a*y_target
        
        sqrt_disc = np.sqrt(discriminant)
        
        x1 = (-b + sqrt_disc) / (2*a)
        x2 = (-b - sqrt_disc) / (2*a)

        return max(x1, x2)
    
    colors = {
        "mab2rec": "#9ec2d7",     # light blue
        "gobrec GPU": "#ffbeb0",  # soft red
        "gobrec CPU": "#bdd0bd",  # light green
        "irec": "#bcbddc"         # light purple
    }
    
    text_colors = {
        "mab2rec": "#0534aa",     # light blue
        "gobrec GPU": "#e50202",  # soft red
        "gobrec CPU": "#277C08",  # light green
        "irec": "#9b1b9f"         # light purple
    }

    df_group = df[df[color] == 'gobrec GPU']

    w, _ = nnls(
        np.hstack(
            (
                (df_group[x].values ** 2).reshape(-1, 1),
                df_group[x].values.reshape(-1, 1)
            )
        ),
        df_group[y].values
    )

    a, b = w

    one_minute_pred = solve(a, b, 60)
    one_hour_pred = solve(a, b, 60*60)

    items_start = np.arange(100, one_minute_pred, (one_minute_pred - 100) / 10)
    items_between = np.arange(one_minute_pred, one_hour_pred, (one_hour_pred - one_minute_pred) / 10)
    items_end = [one_hour_pred, one_hour_pred * 1.03, one_hour_pred * 1.06]
    new_x = np.concatenate(([one_minute_pred], items_between, [one_hour_pred], items_end))
    
    fig = go.Figure()
    
    # Add vlines to indicate 1 minute and 1 hour
    fig.add_vline(x=one_minute_pred, line=dict(color="#dad1af", dash='dash'), layer='below')
    fig.add_vline(x=one_hour_pred, line=dict(color='#dad1af', dash='dash'), layer='below')
    
    for c_group in df[color].unique():
        df_group = df[df[color] == c_group]

        w, _ = nnls(
            np.hstack(
                (
                    (df_group[x].values ** 2).reshape(-1, 1),
                    df_group[x].values.reshape(-1, 1)
                )
            ),
            df_group[y].values
        )
        a, b = w
        
        predictions = a * (new_x ** 2) + b * new_x
        fig.add_trace(
            go.Scatter(
                x=new_x,
                y=predictions,
                name=c_group,
                showlegend=True,
                mode='lines',
                line=dict(color=colors.get(c_group, 'black'))
            )
        )

    for c_group in df[color].unique():
        df_group = df[df[color] == c_group]

        w, _ = nnls(
            np.hstack(
                (
                    (df_group[x].values ** 2).reshape(-1, 1),
                    df_group[x].values.reshape(-1, 1)
                )
            ),
            df_group[y].values
        )
        a, b = w
        y_minute = a * (one_minute_pred ** 2) + b * one_minute_pred
        y_hour = a * (one_hour_pred ** 2) + b * one_hour_pred
        
        if c_group == 'irec':
            text_position = "top center"
        else:
            text_position = "bottom center"

        fig.add_trace(
            go.Scatter(
                x=[one_minute_pred],
                y=[y_minute],
                mode="markers+text",
                text=[f"{round(y_minute / 60)}m"],
                textposition=text_position,
                showlegend=False,
                textfont=dict(color=text_colors.get(c_group, 'black')),
                line=dict(color=colors.get(c_group, 'black'))
            )
        )

        fig.add_trace(
            go.Scatter(
                x=[one_hour_pred],
                y=[y_hour],
                mode="markers+text",
                text=[f"{round(y_hour / 3600)}h"],
                textposition=text_position,
                showlegend=False,
                textfont=dict(color=text_colors.get(c_group, 'black')),
                line=dict(color=colors.get(c_group, 'black'))
            )
        )
    
    fig.update_layout(
        legend_title_text='',
        yaxis_type="log",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        width=600,
        height=500,
        margin=dict(b=10, t=0, r=0, l=0)
    )
    
    return fig

generate_predictions_line_plot(df, x='Items number', y='Mean execution time (s)', color='Algorithm')

In [5]:

def generate_line_plot_with_enhances(df, x, y, color):
    colors = {
        "mab2rec": "#9ec2d7",     # light blue
        "gobrec GPU": "#ffbeb0",  # soft red
        "gobrec CPU": "#bdd0bd",  # light green
        "irec": "#bcbddc"         # light purple
    }
    
    text_colors = {
        "mab2rec": "#0534aa",     # light blue
        "gobrec GPU": "#e50202",  # soft red
        "gobrec CPU": "#277C08",  # light green
        "irec": "#9b1b9f"         # light purple
    }

    fig = go.Figure()
    
    df['enhance'] = (
        df["Mean execution time (s)"] /
        df.groupby("Items number")
        .apply(lambda g: g.loc[g["Algorithm"] == "gobrec GPU", "Mean execution time (s)"].iloc[0])
        .reindex(df["Items number"]).values
    )
    
    df_group = df[df[color] == 'gobrec GPU']
    
    fig.add_vline(x=df_group[x].iloc[1], line=dict(color="#dad1af", dash='dash'), layer='below')
    fig.add_vline(x=df_group[x].iloc[3], line=dict(color='#dad1af', dash='dash'), layer='below')
    
    for c_group in df[color].unique():
        df_group = df[df[color] == c_group]

        fig.add_trace(
            go.Scatter(
                x=df_group[x],
                y=df_group[y],
                name=c_group,
                showlegend=True,
                mode='lines',
                line=dict(color=colors.get(c_group, 'black'))
            )
        )

    for c_group in df[color].unique():
        df_group = df[df[color] == c_group]
        
        if c_group == 'irec':
            text_position = "top center"
        else:
            text_position = "bottom center"

        fig.add_trace(
            go.Scatter(
                x=[df_group[x].iloc[1]],
                y=[df_group[y].iloc[1]],
                mode="markers+text",
                text=[f"{round(df_group[y].iloc[1])}s \\\\ ({round(df_group['enhance'].iloc[1], 1)}x)"],
                textposition=text_position,
                showlegend=False,
                textfont=dict(color=text_colors.get(c_group, 'black')),
                line=dict(color=colors.get(c_group, 'black'))
            )
        )

        fig.add_trace(
            go.Scatter(
                x=[df_group[x].iloc[3]],
                y=[df_group[y].iloc[3]],
                mode="markers+text",
                text=[f"{round(df_group[y].iloc[3])}s \\\\ ({round(df_group['enhance'].iloc[3], 1)}x)"],
                textposition=text_position,
                showlegend=False,
                textfont=dict(color=text_colors.get(c_group, 'black')),
                line=dict(color=colors.get(c_group, 'black'))
            )
        )
    
    max_y = df[y].max()
    min_y = df[y].min()
    fig.update_yaxes(
        range=[
            np.log10(min_y),
            np.log10(max_y) * 1.1
        ]
    )
    fig.update_layout(
        legend_title_text='',
        yaxis_type="log",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="center",
            entrywidth=77,
            x=0.5
        ),
        width=600,
        height=500,
        margin=dict(b=10, t=0, r=0, l=0)
    )

    return fig

generate_line_plot_with_enhances(df, x='Items number', y='Mean execution time (s)', color='Algorithm')

NameError: name 'df' is not defined

In [ ]:
generate_line_plot(df, x='Items number', y='Mean execution time (s)', color='Algorithm', log_y=True, log_x=False)

In [ ]:
df

In [22]:
df['enhance'] = (
    df["Mean execution time (s)"] /
    df.groupby("Items number")
      .apply(lambda g: g.loc[g["Algorithm"] == "gobrec GPU", "Mean execution time (s)"].iloc[0])
      .reindex(df["Items number"]).values
)
df

,Algorithm,Items number,Mean execution time (s),Standard deviation (s),enhance
0,mab2rec,1000,270.219903,1.484943,89.746763
1,gobrec GPU,1000,3.010915,0.022926,1.000000
2,gobrec CPU,1000,19.531463,0.051943,6.486886
3,irec,1000,534.493105,2.675148,177.518478
4,mab2rec,5000,1431.578190,23.649792,147.046918
5,gobrec GPU,5000,9.735520,0.236103,1.000000
6,gobrec CPU,5000,90.373106,1.092226,9.282823
7,irec,5000,1590.560502,21.078239,163.377048
8,mab2rec,10000,2899.482545,33.351406,160.173777
9,gobrec GPU,10000,18.102105,0.279703,1.000000
